<a href="https://colab.research.google.com/github/toyo1867/jankennkyu/blob/main/%E5%B7%9D%E6%9D%91%E8%BB%8D%E5%9B%A3%E8%A7%A3%E6%9E%90_ipynb_%E3%81%AE%E3%82%B3%E3%83%94%E3%83%BC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

このセル（verify_and_create_dataset関数）は、卒論における「データ収集・前処理」の心臓部であり、非常に重要な役割を果たしています。

端的に言うと、「バラバラに圧縮された1万件以上の天鳳の対局ログを、一括で読み込み、Pythonで扱いやすい巨大な『表（DataFrame）』に変換して検証する」という処理を行っています。

ステップごとに解説します。

1. ZIPとGZIPの解凍・解析 (zipfile, gzip)
ファイルを開く: 指定されたZIPファイルの中にある、.mjlog という拡張子のファイルを1つずつ順番に取り出しています。

圧縮の解除: 天鳳のログはさらに .gz 形式で圧縮されているため、gzip.decompress を使って、中身のXMLデータを展開しています。

2. XML構造の読み込み (ElementTree)
XMLパーサー: ET.fromstring(content) を使い、XML形式のログをプログラムが理解できる「木の構造（タグ）」に変換しています。

全イベントの記録: 局の開始（INIT）、ツモ、打牌、和了など、XMLに含まれるすべての要素（タグ）を1つずつ順番に追いかけています。

3. 表形式への変換 (pandas)
属性の抽出: XMLの各タグには、属性（例：oya="0", ten="250,250,250,250" など）が付いています。{child.attrib} を使って、この属性を「列」として、その時の tag_type と file_id（どのファイルのデータか）を「値」として記録しています。

DataFrame作成: 全ファイルから集めたすべての行（all_rows）を、最後に一気に pd.DataFrame(all_rows) で巨大な表にしています。

4. 網羅性の検証 (nunique)
ファイル漏れチェック: unique_files = df['file_id'].nunique() を実行し、データフレームの中に何種類の file_id が存在するかを確認しています。

目標との照合: 「合計11,573個のファイル」がすべて処理されたかどうかを最後に判定し、データに抜け漏れがないことを証明しています。

なぜこの処理が重要なのか？
通常、1万件ものファイルを1つずつ人間が確認するのは不可能です。このコードは以下のことを自動で行っています。

一貫性: すべてのファイルを同じルールで変換するため、分析時のデータに偏りがありません。

網羅性: 1件のログ漏れも許さない検証工程が含まれているため、卒論の「分析の信頼性」を支えています。

可搬性: 1,200万行を超える生のXMLデータを、1つのメモリ（Pandas DataFrame）に乗せられる形式に変換して、すぐに統計分析や機械学習が可能な状態にしています。

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
#XML解析、データ表作成、ZIP/GZIP圧縮ファイル操作に必要なライブラリのインポート
import xml.etree.ElementTree as ET
import pandas as pd
import zipfile
import gzip
#関数の定義
def verify_and_create_dataset(zip_path):
  #抽出した全データを一時的にためるからのリスト（これを最後にDataFrameに変換する）
    all_rows = []
    # ファイルごとの処理数を記録する辞書（正しく処理されたかを確認する）
    process_status = {}

    print("全ファイルの解析を開始します...")
    #指定したZIPファイル全体を読み取り専用モード（'r'）で開く。withによって処理が終わったら自動的にファイルを閉じるため、メモリの節約と安全性が高まる
    #zipfileライブラリのZipFileクラスを使う。zip_pathでどのファイルを開くかを指定する
    with zipfile.ZipFile(zip_path, 'r') as z:
      #z.namelist()：zipの中にあるファイル名のリストを返す
      #for f in z.namelist()：そのリストから、ファイルを一つずつ取り出してfという変数に入れる
      #f if f.endswith('mjlog')：そのファイル名fが.mjlogで終わるならリストの要素として採用する
        file_list = [f for f in z.namelist() if f.endswith('.mjlog')]
        #リストアップした対局ログを、一つずつ順番に取り出して処理する
        for file_name in file_list:
          #file_nameのファイルを一つ指定して開く
            with z.open(file_name) as f:
              #ZIPの中にGZIPという二重圧縮の状態で入っているため、decompressで解凍する。上だけではファイルを開けただけで、中身はない
              #→f.read()でファイルの中身を順に読み取り、バイトデータ（01の塊）としてpythonにコピーしてくる
                content = gzip.decompress(f.read())
                #解凍したテキスト（XML）を、ElementTreeライブラリで「プログラムが操作可能な木の構造」に変換する
                root = ET.fromstring(content)

                count = 0
                #XMLの構造に従ってタグを一つずつ順番に見る
                for child in root:
                    # child.attrib:タグが持っている属性（詳細設定）の塊。<INIT oya="0" ten="250,250" /> というタグなら、{"oya": "0", "ten": "250,250"} という辞書が入っています。
                    #**で、辞書の中身をバラバラに展開する→oya='0'という情報がrecordの中に入る
                    #tag_typeで
                    record = {**child.attrib, 'tag_type': child.tag, 'file_id': file_name}
                    all_rows.append(record)
                    count += 1

                # このファイルの処理数を記録
                process_status[file_name] = count

    # 全件数とファイル数の検証
    df = pd.DataFrame(all_rows)
    unique_files = df['file_id'].nunique()

    print("-" * 30)
    print(f"検証結果:")
    print(f"処理した合計ファイル数: {unique_files} (目標: 11573)")
    print(f"総イベント数: {len(df)}")

    # もし11573未満なら、どのファイルが処理されていないか特定する
    if unique_files < 11573:
        print("警告: すべてのファイルが処理されていません。")
    else:
        print("成功: 11573個の全ファイルからデータを正常に取得しました。")

    return df

# 実行
df = verify_and_create_dataset('/content/drive/MyDrive/麻雀研究/mjlog_pf4-20_n9.zip')

全ファイルの解析を開始します...
------------------------------
検証結果:
処理した合計ファイル数: 9521 (目標: 11573)
総イベント数: 8058948
警告: すべてのファイルが処理されていません。


1. データの基本構造の確認 (len, columns)
やっていること: 全データが何行あるか、そしてデータフレームにはどんな項目（列）が用意されているかを表示します。

目的: 「正しく1万件以上の全ファイルが取り込めているか（行数が想定通りか）」と、「分析に必要な列（例：tag_type, file_id, pai など）がちゃんと存在しているか」をチェックします。

2. タグの種類ごとの件数 (value_counts)
やっていること: データの中に含まれる「イベントのタイプ（tag_type）」ごとに、それぞれ何回ずつ出現しているかを集計します。

目的: データの「内訳」を知るためです。

例えば、「リーチ」が17万回、「和了」が10万回あるなら、「ああ、このデータセットは対局の主要なアクションを網羅できているな」と判断できます。

また、N というタグが多すぎるなど、「分析に邪魔なノイズ（不要なシステムログ）がどの程度あるか」を確認するのにも使います。

3. 実際のデータのサンプル (head)
やっていること: データの「最初の3行」だけを画面に表示します。

目的: 列の中身がどういうフォーマット（文字か、数字か、NaNという空のデータか）になっているか、自分の目で確認するためです。プログラムで処理する前に、まずは生の状態を目視で把握する「直感的な確認」です。

4. 特定の局面データのチェック (reach_data)
やっていること: 数百万行あるデータの中から、特定のイベント（ここでは「リーチ」）だけに絞り込んで、その詳細な内容を1行表示します。

目的: 「本当にリーチした瞬間の情報が、プログラムで扱える形で抽出できているか」を確かめる「動作確認」です。

卒論におけるこのセルの立ち位置
このセルは、いわば「データの解像度を調整するフェーズ」です。

もしここで「あれ、リーチが0件しかないぞ？」となれば、読み込み処理（前のセル）に戻って、何が失敗しているかを修正する必要があります。

もしここで「列が多すぎて何が何だか分からない」となれば、必要な列だけに絞り込むという次の方針が立てられます。

In [5]:
# 1. データの基本構造の確認
print("--- データの基本情報 ---")
print(f"データフレームの行数: {len(df)}")
print(f"列一覧: {list(df.columns)}")

# 2. タグの種類（何が記録されているか）の確認
print("\n--- タグの種類ごとの件数（tag_type） ---")
print(df['tag_type'].value_counts().head(20)) # 上位20個を表示

# 3. 実際のデータのサンプル（どんなデータが入っているか）
print("\n--- データのサンプル（最初の3件） ---")
print(df.head(3))

# 4. 「特定の局面」のデータがちゃんと取れているかのチェック
# 例：リーチ(REACH)が記録されている行を確認
reach_data = df[df['tag_type'] == 'REACH']
print(f"\n--- REACHイベントの数: {len(reach_data)}件 ---")
if len(reach_data) > 0:
    print(reach_data.head(1))

--- データの基本情報 ---
データフレームの行数: 8058948
列一覧: ['seed', 'ref', 'tag_type', 'file_id', 'type', 'lobby', 'n0', 'n1', 'n2', 'n3', 'dan', 'rate', 'sx', 'oya', 'ten', 'hai0', 'hai1', 'hai2', 'hai3', 'who', 'step', 'ba', 'hai', 'machi', 'yaku', 'doraHai', 'doraHaiUra', 'fromWho', 'sc', 'm', 'owari', 'yakuman', 'paoWho']

--- タグの種類ごとの件数（tag_type） ---
tag_type
N        178400
REACH    105056
INIT      81314
AGARI     70862
G119      11929
E122      11918
D117      11901
D119      11900
F121      11900
E120      11890
G116      11889
F123      11883
F122      11867
F116      11852
D121      11823
E121      11816
D118      11811
G123      11810
G117      11804
E117      11799
Name: count, dtype: int64

--- データのサンプル（最初の3件） ---
                                                seed  ref tag_type  \
0  mt19937ar-sha512-n288-base64,c9q5aqR9ACiF9tsl6...       SHUFFLE   
1                                                NaN  NaN       GO   
2                                                NaN  NaN       UN   

1. 和了データ（df_agari）の読み解き方
yaku (役コード):
ここにはカンマ区切りの数字が入っています。例えば 52 があれば「リーチ」、1 があれば「ツモ」です。

卒論での活用: 頻出する役コードをランキング化することで、しょぼんnさんが「リーチ派」なのか「鳴き手派」なのか、「高打点狙い」なのか「速攻狙い」なのかという、プレイスタイルの定量的評価が可能です。

sc (得点状況):
250, 0, 250, 10 のような数字は、その時の4人の持ち点です。

卒論での活用: 「トップ目の時の和了率」や「ラス目の時の和了率」を比較することで、点数状況に応じたリスク管理能力を分析できます。

2. リーチデータ（df_reach）の読み解き方
step:
リーチにはステップがあります。step 1（宣言）と step 2（供託支払い）というセットで記録されています。

who:
誰がリーチしたかを表すIDです。

卒論での活用: 自分のID（しょぼんnさん）の行だけを抽出することで、「どの手牌の状況でリーチ判断をしたか（何巡目か、ドラはどれか）」を逆引きできます。

In [6]:
# 和了（AGARI）のデータにだけ絞って確認
df_agari = df[df['tag_type'] == 'AGARI']
print("和了データのサンプル（点数や役が入っているはず）:")
print(df_agari[['tag_type', 'yaku', 'sc', 'doraHai']].head())

# リーチ（REACH）のステップ確認
df_reach = df[df['tag_type'] == 'REACH']
print("\nリーチデータのサンプル（リーチのステップ数が入っているはず）:")
print(df_reach[['tag_type', 'step', 'who']].head())

和了データのサンプル（点数や役が入っているはず）:
    tag_type                    yaku                               sc doraHai
55     AGARI       1,1,2,1,14,1,53,1       250,0,240,90,250,0,250,-80      39
170    AGARI  1,1,0,1,52,1,54,1,53,1  250,-40,320,140,240,-40,170,-40     102
284    AGARI                8,1,54,1       210,0,460,0,200,23,130,-23  36,129
388    AGARI  1,1,7,1,52,1,54,2,53,0        210,-80,460,0,223,0,97,90      90
519    AGARI                     0,1      130,-3,460,-3,223,21,177,-5     127

リーチデータのサンプル（リーチのステップ数が入っているはず）:
    tag_type step who
48     REACH    1   1
50     REACH    2   1
146    REACH    1   1
148    REACH    2   1
162    REACH    1   2


「代表選手の選抜」:
df['file_id'].unique()[0] で、数万試合あるデータセットの中から「最初の1試合」だけを代表として選んでいます。

「物語の再現」:
元のデータは数百万行が混ざった「辞書」のような状態ですが、match_df に絞り込むことで、「ある1人のプレイヤーが、ある日対局した試合」の全記録を切り出しています。

「時系列の可視化」:
iterrows() を使って、対局開始から終了までのイベントを上から順に表示しています。これにより、「どんな牌がツモられ、誰が切ったか（打牌したか）」という対局のリアルな時間経過を追体験できます。

In [7]:
# 1. 特定の試合を1つ取り出す
sample_file = df['file_id'].unique()[0] # 最初の試合
match_df = df[df['file_id'] == sample_file]

# 2. 試合の流れを時系列で見る（タグの出現順）
# 読み込み順がそのまま時系列なので、そのまま表示すればOK
print(f"--- 試合ID: {sample_file} の流れ ---")
for index, row in match_df.iterrows():
    print(f"イベント: {row['tag_type']}, 詳細: {row.to_dict()}")
    # ※多すぎるので実際には件数を制限して見てください
    if index > 10: break

--- 試合ID: mjlog_pf4-20_n9/2012112403gm-0001-0000-91426bc4&tw=1.mjlog の流れ ---
イベント: SHUFFLE, 詳細: {'seed': 'mt19937ar-sha512-n288-base64,c9q5aqR9ACiF9tsl6+iCJXrKgiwn5YqDH8svhijv4ulXyDhRyWJJmf0bu1T0KhIDg51yqT3DcyAlbaIgQgNyFChQbem+83//zccuevOlgB+a9xQw0kbqZ8cofOQ8BN1i6RbuFS8E3bKdR5kLEbY5RrnErVRbEmkqCXBnrbByQoIlNAGs6tC1iQqKNUPclNKb5FBFWydFmVTLm7m6CpmgxNPv2H2ONc1mV7BKqyRarC7KoSQa+AKgslzyogrTLdbkVgNzfyaU5WLZRuoG/Gbg+LDBazixAhY7byqKOpbjgU4yQDKYlYbLktKauoC8CeQiQN0kTVqKnORsFPgsLLB1+8Xc+OqqJqA5ILIvDN3IwlV1fzNEt9pJjKVhzTQGboDJ+lsJXQsWsZEef6yoeo0ctA+0WsR/90kd4zvO8t1rQ83leHUqtBy7snSxGYYppz7HcOW9c9hl1Y1ToavmP6YGM06ce/xoOZs1E904t8fN0QQwbaatCMaogrYpg3/uKqg0eS3y3yKGOq5eW4TEmKrHUp45DrYejUJ4ycA0UW58cnOuAK+Vofipi3bkM1FZnmK5Nh08dOntlqOU2TCTEymd2YtSdzvo/APSQL9WGTKuuDvmW0uU8H1zwm5Pb905vbeMkbC2du9WjlrJ4bavKUAjDtGbTB0rAiuHf2EVxkR93G7qvoeD0rCLqkzAMDn0TD9fSaGm1vErIxZYykqjrJSdwpI88ztFb9IQP10C98Xo2vKMm4cuVM7cC2VhJ+G9HUbd8joxIRtx1k+2EnHmc57SysRr5xBd95CD8A5gsyKazCFuC+48TV6OLH9He4zzrUCEeKgFM968T6H/SwqcoXqCJOMkEqud32g3f

1. このコード（structure_match_data）がしていること
このコードは、バラバラで複雑な「イベントログ（XML）」を、分析に適した「1行1イベントのフラットな表（DataFrame）」に変換する変換器です。

目的: 試合の「局（Kyoku）」単位での状態を、打牌アクションごとに紐付けること。

主な処理プロセス:

試合ごとのループ: df.groupby('file_id') で、何千もの試合を1つずつ順番に取り出します。

局情報のキャッシュ: current_game_info という辞書を作り、その局の開始時（INITタグ）に「誰が親か」「配牌は何か」を一時的に記憶（キャッシュ）します。

状態の付与: 打牌（U系やD系）のイベントが発生するたびに、その時点まで保持していた「その局の初期情報」を1行のデータとして強引に合体させます。

構造化: これにより、「その打牌をした時、配牌はどうだったか」という情報が、1行で確認できるようになります。

2. 出力結果が意味していること
出力された表は、「対局の歴史を、打牌の瞬間に絞って抽出したスナップショット」です。

列の説明:

file_id: そのデータがどの対局ログから来たかの識別子。

tag_type: 発生したアクションの種類（D120 は「120番の牌を捨てた」というアクション）。

pai: 捨てられた牌の数値コード。

oya: その局の親のID（0=東家など）。

hai0〜hai3: ここが重要です。 その局が始まった瞬間の4人の配牌情報が、打牌データと並んで記録されています。

データの見方（例：1行目）:

file_id: mjlog... の対局の、

tag_type: D120 というアクションで、

pai: 120 という牌を捨てた。

hai0〜hai3: その局の開始時、全員の手牌はこうだった。

In [8]:
import pandas as pd

def structure_match_data(df):
    structured_rows = []

    # 試合ごとに処理
    for file_id, match_df in df.groupby('file_id'):
        current_game_info = {}

        # 行の順序を保証するためにソート（もしインデックスがバラバラの場合のため）
        for _, row in match_df.sort_index().iterrows():
            # 局開始時に情報を更新
            if row['tag_type'] == 'INIT':
                current_game_info = {
                    'oya': row['oya'],
                    'hai0': row['hai0'], 'hai1': row['hai1'],
                    'hai2': row['hai2'], 'hai3': row['hai3']
                }

            # 打牌(U)やリーチ(D)の場合、その時点の状態を記録
            elif isinstance(row['tag_type'], str) and row['tag_type'].startswith(('U', 'D')):
                record = {
                    'file_id': file_id,
                    'tag_type': row['tag_type'],
                    'who': row.get('who'), # 誰が切ったか
                    'pai': row['tag_type'][1:], # 切った牌のコード
                    **current_game_info # その局の初期状態を付与
                }
                structured_rows.append(record)

    return pd.DataFrame(structured_rows)

# --- 実行セクション ---
# 最初の1ファイル分だけ取り出してテスト
sample_file_id = df['file_id'].unique()[0]
sample_df = df[df['file_id'] == sample_file_id].copy() # .copy()を追加してSettingWithCopyWarningを回避

print(f"解析中: {sample_file_id}...")
df_structured = structure_match_data(sample_df)

print(f"構造化完了！抽出件数: {len(df_structured)}")
print(df_structured.head())

解析中: mjlog_pf4-20_n9/2012112403gm-0001-0000-91426bc4&tw=1.mjlog...
構造化完了！抽出件数: 129
                                             file_id tag_type  who  pai  oya  \
0  mjlog_pf4-20_n9/2012112403gm-0001-0000-91426bc...       UN  NaN    N  NaN   
1  mjlog_pf4-20_n9/2012112403gm-0001-0000-91426bc...     D115  NaN  115    0   
2  mjlog_pf4-20_n9/2012112403gm-0001-0000-91426bc...      U24  NaN   24    0   
3  mjlog_pf4-20_n9/2012112403gm-0001-0000-91426bc...      D74  NaN   74    0   
4  mjlog_pf4-20_n9/2012112403gm-0001-0000-91426bc...     U124  NaN  124    0   

                                         hai0  \
0                                         NaN   
1  30,58,115,0,74,44,126,102,132,135,29,59,71   
2  30,58,115,0,74,44,126,102,132,135,29,59,71   
3  30,58,115,0,74,44,126,102,132,135,29,59,71   
4  30,58,115,0,74,44,126,102,132,135,29,59,71   

                                          hai1  \
0                                          NaN   
1  110,89,50,107,37,105,108,20,47,54,117,

このコードがやっていること
Pandasの apply 関数を使って、データフレームの横方向に（axis=1）、以下の処理を行っています。

df_structured[['hai0', 'hai1', 'hai2', 'hai3']]: 4人分の配牌データが入った4つの列を選択します。

x.dropna(): もし万が一データが欠損している行があっても、計算エラーにならないように取り除きます。

.astype(str): 全ての数字を「文字列」に変換します（カンマで結合できるようにするため）。

','.join(...): カンマでつなげて、1つの長い文字列にします。

In [9]:
# 例：手牌を扱いやすくするために、カンマ区切りをリストに変換
df_structured['hai_combined'] = df_structured[['hai0', 'hai1', 'hai2', 'hai3']].apply(
    lambda x: ','.join(x.dropna().astype(str)), axis=1
)

df_structured['pai']: 先ほど作成した、各打牌アクション（UやDタグ）を記録したテーブルから、「捨てた牌のコード（pai）」という列を指定しています。

.value_counts(): この列に含まれる「数字」がそれぞれ何回ずつ出現したかを数え上げます。

.head(10): 出現回数が多い順に並び替え、トップ10だけを取り出しています。

In [10]:
# しょぼんnさんが最もよく切る牌の集計
print(df_structured['pai'].value_counts().head(10))

pai
4      4
0      4
110    3
68     3
39     3
111    2
115    2
126    2
119    2
103    2
Name: count, dtype: int64


1. 変換のロジック：なぜ // 4 をするのか？
天鳳では、同じ種類の牌が4枚存在します（例：1萬が4枚）。システム上、これらを区別するために0〜135の連番が振られていますが、麻雀の牌の種類としては「4枚で1グループ」です。

idx = code // 4:
コードを4で割ることで、「0〜135の連番」を「0〜33の牌の種類（34種類）」に圧縮しています。

0, 1, 2, 3 を4で割ると、すべて商は 0 になります。つまり、この4つのコードはすべて「1萬」というグループに属することを示しています。

2. IF文による分類（牌の特定）
34種類のインデックス（0〜33）を、麻雀のルールに基づいたグループに振り分けています。

0〜8 (9種類): 萬子（m）。インデックスから1を足して数字を確定させます。

9〜17 (9種類): 筒子（p）。インデックスから9を引いて数字を確定させます。

18〜26 (9種類): 索子（s）。インデックスから18を引いて数字を確定させます。

27〜33 (7種類): 字牌。リスト jihai から、インデックスの差分を使って日本語の名前を取り出します。

In [11]:
def get_pai_name(code):
    # 牌コードから 0-33 のインデックスに変換
    idx = code // 4

    if 0 <= idx <= 8:
        return f"{idx + 1}m"  # 萬子
    elif 9 <= idx <= 17:
        return f"{idx - 9 + 1}p" # 筒子
    elif 18 <= idx <= 26:
        return f"{idx - 18 + 1}s" # 索子
    elif 27 <= idx <= 33:
        jihai = ["東", "南", "西", "北", "白", "發", "中"]
        return jihai[idx - 27]
    return "不明"

# テストしてみる
print(get_pai_name(0))   # 1m
print(get_pai_name(36))  # 1p
print(get_pai_name(108)) # 東

1m
1p
東


1. 「今の（さっきの）と一緒」な点
役割: pai（数字コード）を人間が読める 1m や 東 に変換する目的は同じです。

変換ロジック: code // 4 で種類を特定し、if文で牌名を決定する基本ルールは変わっていません。

2. 「より賢くなっている（進化している）」点
このコードには、「データが汚れていても止まらないための防御力」が備わっています。

try...except ブロック:
これが最も重要です。もし pai 列の中に、予期せぬ文字（'N' だけでなく、空文字や、エラー値など）が入っていても、プログラムが強制終了せず、「その他」というラベルをつけて無視してくれます。

apply(safe_get_pai_name):
1行ずつ安全に判定を行っているため、データセット全体に対して一括で処理をかける際、非常に安定しています。

In [12]:
def safe_get_pai_name(val):
    try:
        # 文字列を整数に変換してから牌名を取得
        code = int(val)
        idx = code // 4
        if 0 <= idx <= 8: return f"{idx + 1}m"
        if 9 <= idx <= 17: return f"{idx - 9 + 1}p"
        if 18 <= idx <= 26: return f"{idx - 18 + 1}s"
        if 27 <= idx <= 33: return ["東", "南", "西", "北", "白", "發", "中"][idx - 27]
    except (ValueError, TypeError):
        # 数字に変換できないもの（'N'など）は除外、または名前を返す
        return "その他"
    return "不明"

# 牌コード列(pai)から安全に牌名列(pai_name)を作成
df_structured['pai_name'] = df_structured['pai'].apply(safe_get_pai_name)

# ランキング確認（"その他"を除外して表示）
print("しょぼんnさんの打牌ランキング（牌名）:")
print(df_structured[df_structured['pai_name'] != "その他"]['pai_name'].value_counts().head(10))

しょぼんnさんの打牌ランキング（牌名）:
pai_name
東     8
1m    8
9p    7
8m    7
2m    6
3s    5
7s    5
白     5
1p    5
7p    5
Name: count, dtype: int64


このコードは、これまで手こずっていた「データのノイズ（数字ではない記号）」を完全に排除し、「打牌の純粋なランキング」を完成させるための最終仕上げです。

1行ずつ、なぜその処理が必要なのかを解説します。

1. pd.to_numeric(..., errors='coerce')
何をしているか: pai列の値を、強引に数値に変換しようとします。

なぜ必要か: データの中には 'N' や他の文字が含まれています。errors='coerce' という設定を使うと、「数字に変換できないものはエラーを出さずに、NaN（欠損値）という『空っぽの箱』に置き換えてね」という命令になります。これによりプログラムが止まらなくなります。

2. df_dapai_only.dropna(subset=['pai_numeric'])
何をしているか: 先ほどの変換で NaN（空っぽの箱）になってしまった行を、データセットからバッサリ切り捨てています。

なぜ必要か: 'N' などのシステムログを分析に混ぜるとランキングが狂います。ここでノイズを物理的に除去することで、データが「純粋な牌のコード」だけになります。

3. astype(int).apply(get_pai_name)
何をしているか: 完全に数字だけになった列を、整数（int）型として扱い、その後に先ほど定義した変換関数（get_pai_name）を適用して、「コードから牌名（1m, 東など）」に翻訳しています。

なぜ必要か: 最後に人間が見て理解できる「麻雀の牌の形式」にするためです。

4. value_counts().head(10)
何をしているか: 変換された「牌名」の出現回数を数え上げ、回数が多い順にトップ10を表示しています。

In [13]:
df_dapai_only = df_structured.copy()

# 1. 'pai'列を数値に変換。数字でないものは NaN になる
df_dapai_only['pai_numeric'] = pd.to_numeric(df_dapai_only['pai'], errors='coerce')

# 2. NaN（変換できなかった行）を削除
df_dapai_only = df_dapai_only.dropna(subset=['pai_numeric'])

# 3. 整数に変換してから牌名変換を適用
df_dapai_only['pai_name'] = df_dapai_only['pai_numeric'].astype(int).apply(get_pai_name)

# 4. 集計
print(df_dapai_only['pai_name'].value_counts().head(10))

pai_name
東     8
1m    8
9p    7
8m    7
2m    6
3s    5
7s    5
白     5
1p    5
7p    5
Name: count, dtype: int64


各列の意味（重要）
tag_type: 「何が起きたか」を示すイベントコードです。

D120: D は「手出し（手牌から捨てた）」、120 は牌のID。

U26: U は「ツモ切り（ツモってきた牌をそのまま捨てた）」、26 は牌のID。

これにより、プレイヤーが「手牌を入れ替えたのか、それとも引いてきた牌をそのまま処理したのか」という意図まで読み取れます。

pai & pai_name:

pai: 天鳳内部の牌コード（0〜135）。

pai_name: あなたが作成した変換関数によって、人間が読める「北」「1s」「9p」といった牌名に変換されたもの。

hai0〜hai3:

これは「その打牌の瞬間に、対局者4人が持っていた手牌情報（の初期値）」です。

これがあるおかげで、「この牌を切った時、他にどんな牌を持っていたのか」を後から再現できます。

hai_combined:

上記 hai0〜hai3 を全てカンマ区切りで結合し、「その局面の全牌情報」としてひとまとめにしたものです。検索処理を高速にするために作りました。

In [14]:
# 最初の10行をすべて表示（列が多いので、pandasの表示設定を一時的に解除します）
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print(df_structured.head(10))

                                             file_id tag_type  who  pai  oya                                        hai0                                         hai1                                      hai2                                 hai3                                       hai_combined pai_name
0  mjlog_pf4-20_n9/2012112403gm-0001-0000-91426bc...       UN  NaN    N  NaN                                         NaN                                          NaN                                       NaN                                  NaN                                                         その他
1  mjlog_pf4-20_n9/2012112403gm-0001-0000-91426bc...     D115  NaN  115    0  30,58,115,0,74,44,126,102,132,135,29,59,71  110,89,50,107,37,105,108,20,47,54,117,111,9  14,36,118,65,25,1,61,92,66,109,18,31,106  3,79,8,6,76,12,91,2,73,114,11,46,43  30,58,115,0,74,44,126,102,132,135,29,59,71,110...        南
2  mjlog_pf4-20_n9/2012112403gm-0001-0000-91426bc...      U24  NaN   24    0  30,58,1

「自分の番号を特定」し、「打牌のたびに手牌を更新する」仕組みを構築します。

In [15]:
def structure_match_data_with_update(df):
    structured_rows = []

    for file_id, match_df in df.groupby('file_id'):
        my_id = None  # 自分が何番目か（0-3）
        current_hand = {} # 4人それぞれの現在の手牌（リストで管理）

        for _, row in match_df.sort_index().iterrows():
            # 1. 自分のIDを特定 (UNタグ等から取得 ※ログ形式に依存)
            # 2. 局開始時の配牌を初期セット
            if row['tag_type'] == 'INIT':
                # 'hai0'〜'hai3'をリストに変換して管理
                current_hand = {
                    0: str(row['hai0']).split(','),
                    1: str(row['hai1']).split(','),
                    2: str(row['hai2']).split(','),
                    3: str(row['hai3']).split(',')
                }

            # 3. 打牌(U, D)のたびに手牌を更新
            elif isinstance(row['tag_type'], str) and row['tag_type'].startswith(('U', 'D')):
                who = int(row.get('who', 0))
                pai_discarded = row['tag_type'][1:] # 切った牌

                # 手牌から切った牌を削除（書き換え）
                if pai_discarded in current_hand[who]:
                    current_hand[who].remove(pai_discarded)

                # 記録（その瞬間の自分の手牌を保存）
                record = {
                    'file_id': file_id,
                    'tag_type': row['tag_type'],
                    'who': who,
                    'current_my_hand': ",".join(current_hand[who]) if who == my_id else None
                }
                structured_rows.append(record)

    return pd.DataFrame(structured_rows)

In [17]:
# もし 0, 1, 2, 3 の順番で打牌が来ていると仮定できる場合
# 行ごとに 0, 1, 2, 3 を順番に割り振る
import numpy as np

# 打牌タグ（UまたはDで始まるもの）だけを抽出して who を補完する
mask = sample_df['tag_type'].str.contains('U|D', na=False)
count = mask.sum()
sample_df.loc[mask, 'who'] = [i % 4 for i in range(count)]